In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('bmh')
from pandas.plotting import scatter_matrix
# from latex_extractor import LatexExtractor
# from pdf_conversion import PDFExtractor
import torch
import tf_keras as keras
from transformers import pipeline
import re

In [3]:
latexfiles_df = pd.read_json("/content/drive/MyDrive/AI systems Project/latex_extracted.json")
latexfiles_df

,file_id,file_name,authors,title,text,equations
0,0087b54c-53c1-4fc5-ac2f-f9a5042def8d,sm.tex,[Ilya Karzhemanov],Computations and ML for surjective rational maps,The present note studies $f: ^2 ^2$ with terms...,[\label{diag} \xymatrix{\n& X \ar[dl]_\varphi ...


In [4]:
def prepare_input(row):
    text = row["text"]
    equations = row["equations"]
    combined = f"Paper Text:\n{text}\n\nEquations:\n{equations}"
    return combined

In [5]:
latexfiles_df["input_text"] = latexfiles_df.apply(prepare_input, axis=1)
latexfiles_df

,file_id,file_name,authors,title,text,equations,input_text
0,0087b54c-53c1-4fc5-ac2f-f9a5042def8d,sm.tex,[Ilya Karzhemanov],Computations and ML for surjective rational maps,The present note studies $f: ^2 ^2$ with terms...,[\label{diag} \xymatrix{\n& X \ar[dl]_\varphi ...,Paper Text:\nThe present note studies $f: ^2 ^...


In [6]:
latexfiles_df['input_text']

,input_text
0,Paper Text:\nThe present note studies $f: ^2 ^...


In [32]:
# def make_prompt(text):
#     return (
#         "Summarize the following scientific text with structure:\n"
#         "Core idea, Why it matters, Methodology, Important equations, Results, Limitations, Impact.\n\n"
#         f"{text}"
#     )

In [8]:
from transformers import BartTokenizer, BartForConditionalGeneration

In [9]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [34]:
summarizer = pipeline(
    "text2text-generation",
    model="facebook/bart-large-cnn",
    framework="pt",
    device=-1
)

Device set to use cpu


In [46]:
def make_prompt(text, equations=None):
    eq_text = "\n".join(equations) if equations else "None"
    prompt = f"""
Summarize the following scientific text in **sections** with the titles:
Core Idea, Why it matters, Methodology, Important Equations, Results, Limitations, Impact.

Provide each section clearly, e.g.:

Core Idea:
<text>

Why it matters:
<text>

...

Important Equations:
<list equations here>

Text:
{text}

Equations:
{eq_text}
"""
    return prompt


In [25]:
def clean_input(text):
    # Remove LaTeX delimiters, control chars, and excessive math markup
    text = re.sub(r'\\[a-zA-Z]+', '', text)  # remove LaTeX commands like \frac, \alpha
    text = re.sub(r'[^a-zA-Z0-9\s.,;:()\-+=*/$%]', ' ', text)  # strip unknown symbols
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [47]:


def structured_summary(text, equations):
    prompt = make_prompt(text, equations)
    # Truncate to avoid too long input
    prompt = prompt[:2000]  # adjust length if needed

    try:
        summary_output = summarizer(prompt, max_length=400, min_length=100, do_sample=False)
        summary_text = summary_output[0].get('summary_text') or summary_output[0].get('generated_text')

        # Split into sections
        sections = {}
        current_section = None
        for line in summary_text.split("\n"):
            line = line.strip()
            if not line:
                continue
            # Check if line starts with section
            if ":" in line:
                sec_name, sec_text = line.split(":", 1)
                sec_name = sec_name.strip()
                sec_text = sec_text.strip()
                sections[sec_name] = sec_text
                current_section = sec_name
            elif current_section:
                # append remaining lines to the last section
                sections[current_section] += " " + line

        return sections

    except Exception as e:
        print("Error summarizing:", e)
        return {}

In [51]:
for i, row in latexfiles_df.iterrows():
  print(row["text"])
  print(row["equations"])


The present note studies $f: ^2 ^2$ with terms and the indeterminacy locus $I_f $. We develop an experimental approach, based on some Python programming and Machine Learning, towards the classification of such maps; a couple of new explicit $f$ is constructed in this way. We also prove (via pure projective geometry) that a general non-regular cubic endomorphism $f$ of $^2$ is surjective if and only if the set $I_f$ has cardinality at least $3$. Let $X$ be a complex projective variety and $f: X X$ its rational endomorphism. Denote by $I_f X$ the (closed) indeterminacy locus of $f$. Then $f$ is called if the induced morphism $X I_f X$ is onto. Such maps were introduced and studied --- from the algebro-geometric point of view --- in the paper (with $X$ being a projective space $^n$). The initial motivation for this study came from the optimal control theory and mathematical genetics. Later a relation of surjective rational maps with algebraic statistics and quantization has also been real

In [48]:
structured_summaries = []

for i, row in latexfiles_df.iterrows():
    print(f"Summarizing: {row['file_name']} ...")
    summary_sections = structured_summary(row["text"], row["equations"])
    structured_summaries.append({
        "file_name": row["file_name"],
        "Structured Summary": summary_sections
    })




Summarizing: sm.tex ...


In [49]:
result_df = pd.DataFrame(structured_summaries)
result_df.to_csv("structured_summaries.csv", index=False)


In [43]:
result_df['Structured Summary']

,Structured Summary
0,{'The present note studies $f': '^2 ^2$ with t...
